使用 spaCy 进行 NER

In [1]:
import spacy
# 加载英文模型
nlp = spacy.load("en_core_web_sm")
# 处理文本
text = "Apple is looking at buying U.K. startup for $1 billion"
doc = nlp(text)
# 输出识别结果
for ent in doc.ents:
    print(ent.text, ent.label_)
'''Apple -> ORG (Organization 组织)。模型知道这里的 Apple 是“苹果公司”，而不是水果苹果。
U.K. -> GPE (Geopolitical Entity 地缘政治实体)。即识别出了“英国”这个国家。
$1 billion -> MONEY (金额)。提取出了“10亿美元”这个数字。'''

Apple ORG
U.K. GPE
$1 billion MONEY


基于规则的简单 NER 实现

In [2]:
import re  #基于规则（正则表达式）的命名实体识别（NER）

def rule_based_ner(text):
    # 匹配日期（格式：MM/DD/YYYY 或 MM-DD-YYYY）
    dates = re.findall(r'\d{1,2}[/-]\d{1,2}[/-]\d{2,4}', text)   #看到这种数字格式就是日期，看到这种符号就是钱，直接抓出来。”
    # 匹配货币（格式：$ 后跟数字）
    currencies = re.findall(r'\$\d+\.?\d*', text)
    return {"日期": dates, "货币": currencies}

# 测试
sample = "会议定于12/15/2023举行，预算为$5000"
print(rule_based_ner(sample))

{'日期': ['12/15/2023'], '货币': ['$5000']}


补充 1：spaCy 支持的所有实体类型

In [3]:
import spacy

# 加载模型
nlp = spacy.load("en_core_web_sm")

# 获取所有实体标签及其描述
labels = nlp.get_pipe('ner').labels
for label in labels:
    print(label)
'''PERSON	人名	Elon Musk, 张三, Stephen
ORG	组织机构	Apple, Microsoft, 联合国, 曼城俱乐部
GPE	地缘政治实体 (国家、城市、州)	U.K., China, Paris, 北京
LOC	地点 (非GPE，如山川、河流、街道)	Rocky Mountains, 密西西比河
DATE	日期/时间范围	12/15/2023, today, next week
MONEY	货币金额	$1 billion, 100美元, 500万元
PERCENT	百分比	50%, 20 percent
CARDINAL	基数词 (纯粹的数字/数量)	1, 2, 3, 5000
ORDINAL	序数词	first, second, third
NORP	国籍、宗教、政治团体	Chinese, Christians, Democrats
FAC	设施 (建筑、机场、车站等)	Heathrow Airport, Eiffel Tower
EVENT	事件 (台风、体育赛事等)	World Cup, Earthquake
LAW	法律、条约	Copyright Law, Treaty of Versailles
PRODUCT	产品	iPhone, Coca-Cola
LANGUAGE	语言	English, Mandarin'''

CARDINAL
DATE
EVENT
FAC
GPE
LANGUAGE
LAW
LOC
MONEY
NORP
ORDINAL
ORG
PERCENT
PERSON
PRODUCT
QUANTITY
TIME
WORK_OF_ART


补充 2：更完整的 spaCy NER 示例

In [4]:
import spacy

nlp = spacy.load("en_core_web_sm")
text = "Elon Musk founded SpaceX in 2002 and Tesla in 2003."
doc = nlp(text)

for ent in doc.ents:
    print(f"实体: {ent.text}")
    print(f"  类型: {ent.label_}")
    print(f"  起始位置: {ent.start_char}")
    print(f"  结束位置: {ent.end_char}")
    """ent.start_char：这个实体在原始字符串中的起始字符索引（从 0 开始算）。
ent.end_char：这个实体在原始字符串中的结束字符索引。"""
    print()

实体: Elon Musk
  类型: PERSON
  起始位置: 0
  结束位置: 9

实体: 2002
  类型: DATE
  起始位置: 28
  结束位置: 32

实体: Tesla
  类型: ORG
  起始位置: 37
  结束位置: 42

实体: 2003
  类型: DATE
  起始位置: 46
  结束位置: 50



补充 3：使用 NLTK 进行 NER

In [1]:
import nltk

# 设置数据搜索路径为当前项目下的 nltk_data 文件夹
nltk.data.path.append(r"./nltk_data")  # 或者写绝对路径 D:/11/NLP/nltk_data

# 此时就不需要再运行 download 了，直接使用
from nltk import pos_tag, ne_chunk
from nltk.tokenize import word_tokenize

text = "Elon Musk founded SpaceX in 2002."
tokens = word_tokenize(text)
pos_tags = pos_tag(tokens)
'''pos_tag(tokens)：词性标注（POS Tag）。识别每个单词是名词、动词、还是介词。
比如这里输出 Elon/NNP（专有名词），founded/VBD（动词过去式），in/IN（介词）。'''
ner_tree = ne_chunk(pos_tags)

print(ner_tree)

(S
  (PERSON Elon/NNP)
  (PERSON Musk/NNP)
  founded/VBD
  (ORGANIZATION SpaceX/NNP)
  in/IN
  2002/CD
  ./.)


补充 4：使用 Stanford NER（通过 NLTK）

In [3]:
import os
from nltk.tag import StanfordNERTagger
'''使用 NLTK 接口来调用 Stanford NER（斯坦福命名实体识别器），对文本进行实体提取。'''
# ==================================================
# 1. 配置Stanford NER文件路径
# ==================================================
stanford_ner_dir = r"D:\11\NLP\data\stanford-ner\stanford-ner-2020-11-17"
jar_path = os.path.join(stanford_ner_dir,"stanford-ner.jar")
model_path = os.path.join(stanford_ner_dir,"classifiers", "english.all.3class.distsim.crf.ser.gz")

# ==================================================
# 2. 检查文件
# ==================================================
if not os.path.isfile(jar_path):
    raise FileNotFoundError(    f"没有找到JAR文件：\n{jar_path}")

if not os.path.isfile(model_path):
    raise FileNotFoundError(   f"没有找到模型文件：\n{model_path}" )
print("JAR文件检查成功")
print("模型文件检查成功")


# ==================================================
# 3. 创建Stanford NER识别器
# ==================================================
st = StanfordNERTagger( model_filename=model_path, path_to_jar=jar_path, encoding="utf-8", java_options="-mx1000m")
'''java_options="-mx1000m"：这行非常关键！因为 Stanford NER 是一个 Java 程序，NLTK 只是作为“翻译官”去调用它。这个参数的意思是分配 1000MB 的堆内存给 Java 虚拟机（JVM）。如果内存给得太小，处理长文本时 Java 进程会崩溃。'''

# ==================================================
# 4. 准备测试文本
# ==================================================
text = "Elon Musk founded SpaceX in 2002."
# StanfordNERTagger接收的是单词列表
tokens = text.replace(".", " .").split()
print("\n分词结果：")
print(tokens)

# ==================================================
# 5. 执行命名实体识别
# =================================================
result = st.tag(tokens)
print("\n命名实体识别结果：")
for word, entity_type in result:
    print(f"{word:<12} {entity_type}")

JAR文件检查成功
模型文件检查成功

分词结果：
['Elon', 'Musk', 'founded', 'SpaceX', 'in', '2002', '.']

命名实体识别结果：
Elon         PERSON
Musk         PERSON
founded      O
SpaceX       ORGANIZATION
in           O
2002         O
.            O


补充 5：使用 Hugging Face Transformers 进行 NER

In [1]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline

# ============================================================
# 1. 指向你刚才存放下载文件的本地文件夹
# ============================================================
model_path = r"D:\11\NLP\data\bert-base-ner-local"
# ============================================================
# 2. 从本地加载（加 local_files_only=True 强制离线）
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(model_path,local_files_only=True)
model = AutoModelForTokenClassification.from_pretrained( model_path,local_files_only=True)
'''AutoTokenizer 和 AutoModelForTokenClassification 是 Hugging Face 的“万能插件”，它会自动识别本地文件夹里存的是什么架构的模型（这里存的是微调过的 bert-base 模型）。
'''
# ============================================================
# 3. 创建 NER pipeline
# ============================================================
ner_pipeline = pipeline("ner",model=model,tokenizer=tokenizer)
'''这是 Hugging Face 特意为新手和工程化做的一个“高级封装”。
如果没有这行，你需要自己写 40 多行代码去处理：把中文切分 -> 转换为数字 ID -> 输入显卡计算 -> 解析输出的矩阵 -> 把数字 ID 翻译回字母 -> 重新合并切碎的单词。pipeline 这行代码一键帮你把所有杂活干完了。'''
# ============================================================
# 4. 测试
# ============================================================
text = "Elon Musk founded SpaceX in 2002 and Tesla in 2003."
results = ner_pipeline(text)
'''在 NER 中，模型会给句子里的每个词打标签：
有标签的实体（如 Elon Musk 打 PER，SpaceX 打 ORG）会被 pipeline 提取出来打印。
非实体（通常简称为 O，即 Outside）：像 founded（动词）、in（介词）、2002（普通数字），它们不属于人名、组织、地点等实体，因此会被 pipeline 默认直接过滤掉，不输出。'''
for result in results:
    print(f"实体：{result['word']}，类型：{result['entity']}，置信度：{result['score']:.4f}")
    '''## 是什么？ 这是 BERT 类模型使用的 WordPiece 分词器的特有标记。代表这个词是前一个词的后缀（子词）。
例如：El + ##on = Elon；Space + ##X = SpaceX。
B- 和 I- 是什么？ 这是标准的 BIO 实体标注法。
B- (Begin)：实体的开头。
I- (Inside)：实体的内部/延续。
O (Outside)：非实体（这里没有显示出来）。
正确情况示例：一个完整人名 Elon Musk 应该被标记为 B-PER, I-PER, I-PER（如果有 3 个词）。'''

C:\Users\Administrator\.conda\envs\rl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Administrator\.conda\envs\rl\lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
Some weights of the model checkpoint at D:\11\NLP\data\bert-base-ner-local were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from 

实体：El，类型：B-PER，置信度：0.5563
实体：##on，类型：I-ORG，置信度：0.5188
实体：Mu，类型：I-PER，置信度：0.7687
实体：##sk，类型：I-PER，置信度：0.5782
实体：Space，类型：B-ORG，置信度：0.9994
实体：##X，类型：I-ORG，置信度：0.9992
实体：Te，类型：B-ORG，置信度：0.9975
实体：##sla，类型：I-ORG，置信度：0.9670


In [4]:
tokenizer = AutoTokenizer.from_pretrained(model_path,local_files_only=True)
model = AutoModelForTokenClassification.from_pretrained( model_path,local_files_only=True)

ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer, grouped_entities=True)

text = "Elon Musk founded SpaceX in 2002 and Tesla in 2003."
results = ner_pipeline(text)

for result in results:
    print(f"实体: {result['word']}, 类型: {result['entity_group']}, 置信度: {result['score']:.4f}")


Some weights of the model checkpoint at D:\11\NLP\data\bert-base-ner-local were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


实体: El, 类型: PER, 置信度: 0.5563
实体: ##on, 类型: ORG, 置信度: 0.5188
实体: Musk, 类型: PER, 置信度: 0.6735
实体: SpaceX, 类型: ORG, 置信度: 0.9993
实体: Tesla, 类型: ORG, 置信度: 0.9823


补充 6：中文 NER（使用 spaCy 中文模型）

In [1]:
import spacy

# 加载中文模型
nlp_zh = spacy.load("zh_core_web_sm")

# 中文 NER 示例
text_zh = "马云是阿里巴巴的创始人，出生于1964年。"
doc_zh = nlp_zh(text_zh)

for ent in doc_zh.ents:
    print(f"实体: {ent.text}, 类型: {ent.label_}")

实体: 马云, 类型: PERSON
实体: 阿里巴巴, 类型: ORG
实体: 1964年, 类型: DATE
